# 00 — Data preparation

Prepare the customer table and build the four-entity database from the source
worksheet `DadosV3`. The source dataset is included as `data/raw.xlsx`.


In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/raw.xlsx").is_file()
             and (p / "src/analysis.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.pipeline import load_data, preparation_tables, monthly_tables, save_tables

raw, prepared, members = load_data()
OUT = ROOT / "results"


The preparation retains all customer records, selects 33 source fields
and adds two date fields. The complete calculations are in
[src/analysis.py](../src/analysis.py), function `prepare`.


In [2]:
pd.DataFrame({
    "Stage": ["Source", "Prepared"],
    "Customer records": [len(raw), len(prepared)],
    "Fields": [len(raw.columns), len(prepared.columns)],
})

,Stage,Customer records,Fields
0,Source,14942,47
1,Prepared,14942,35


`FirstAnniversaryDate` adds 12 calendar months to the joining date.
`ObservedTenureDays` is the number of calendar days between the first and final
recorded dates. Date calculations use dates without time-of-day components.


In [3]:
prepared[["Id", "EnrollmentStart", "EnrollmentFinish",
          "FirstAnniversaryDate", "ObservedTenureDays"]].head()

,Id,EnrollmentStart,EnrollmentFinish,FirstAnniversaryDate,ObservedTenureDays
0,1C7DEF50-F9F3-4B51-BBD8-35C8511F5FBE,2019-09-03 10:05:03.020,2019-10-31,2020-09-03,58
1,804B9C2B-DD94-4CBF-BA18-510AE12F92E7,2014-08-12 11:59:51.840,2015-09-14,2015-08-12,398
2,9ABB61F3-81BA-446E-927F-3879BF62688A,2017-05-02 09:43:11.320,2017-06-01,2018-05-02,30
3,C47452B2-29C5-4D6B-A28A-A2273D4158FA,2018-09-05 15:30:07.777,2019-02-12,2019-09-05,160
4,EAFD54FD-CAFC-4DA8-8926-6ED0463F02DF,2016-04-20 18:33:21.170,2018-06-07,2017-04-20,778


Source checks cover identifiers, dates, status, renewal counts and
activity indicators. Unknown ages are excluded from the age comparison only.
`REVIEW` identifies source inconsistencies retained for inspection.


In [4]:
tables = preparation_tables(prepared, members)
tables["SourceChecks"]

,Check,AffectedRecords,Severity,Result
0,Missing customer IDs,0,ERROR,PASS
1,Duplicate customer IDs,0,ERROR,PASS
2,Missing or invalid EnrollmentStart,0,ERROR,PASS
3,Missing or invalid EnrollmentFinish,0,ERROR,PASS
4,Missing essential dates or status,0,ERROR,PASS
5,Invalid Dropout coding,0,ERROR,PASS
6,Finish precedes start,0,ERROR,PASS
7,Dates beyond source cutoff,0,ERROR,PASS
8,Uncensored records not at cutoff,0,ERROR,PASS
9,Censoring within reporting period,0,ERROR,PASS


Save the prepared tables and build the database described in
[database/schema.sql](../database/schema.sql). `Member`, `Membership` and
`MembershipStatus` have one row per customer. `Activity` has one row per
customer/activity combination. The build checks keys and database integrity.


In [5]:
from src.database import build_database

save_tables(OUT / "data", tables)
database = build_database(members, OUT / "database", ROOT / "database/schema.sql")
pd.DataFrame(database["tables"].items(), columns=["Table", "Rows"])

,Table,Rows
0,Member,14942
1,Membership,14942
2,MembershipStatus,14942
3,Activity,16372
